# Yvy PMTiles — GPKG → PMTiles (todo en sesión)

Genera los 7 PMTiles de `08_pmtiles/` desde `01_work/*.gpkg`. Todo se instala
en la sesión (nada cacheado en Drive): tippecanoe **2.79.0** compilado + geopandas.

**Contrato (SPEC §10b, no tocar):** cada feature lleva propiedad `id` = `yvy_id`
(`--use-attribute-for-id=id`), un archivo por nivel/tipo, nombre = nivel o tipo.
El repo verifica el match en `pnpm run build` (avisa `X/Y con id coincidente`).

In [ ]:
# 1) tippecanoe 2.79.0 desde source (pin SPEC) + geopandas + Drive
import subprocess, sys
print(subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch', '2.79.0',
                      'https://github.com/felt/tippecanoe.git', '/content/tippecanoe']).returncode)
!apt-get update -qq && apt-get install -y -qq build-essential libsqlite3-dev zlib1g-dev
!cd /content/tippecanoe && make -j$(nproc) && make install
!tippecanoe --version
%pip install -q geopandas pyogrio
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Config: GPKG -> capas -> PMTiles (zooms = headers actuales del repo)
from pathlib import Path
BASE = Path('/content/drive/MyDrive/YvyLabs/YvyExplorerProduction')
WORK, OUT = BASE / '01_work', BASE / '08_pmtiles'
OUT.mkdir(parents=True, exist_ok=True)

# (gpkg, capa, salida, zmin, zmax)
PLAN = [
    ('nacional.gpkg',   'pais',          'paises.pmtiles',        0, 8),
    ('provincial.gpkg', 'provincias',    'provincias.pmtiles',    4, 10),
    ('provincial.gpkg', 'departamentos', 'departamentos.pmtiles', 6, 12),
    ('municipal.gpkg',  'municipios',    'municipios.pmtiles',    8, 14),
    ('local.gpkg',      'barriosSalCap', 'barrios.pmtiles',       10, 16),
    ('provincial.gpkg', 'catMinero',     'mineria.pmtiles',       8, 14),
    ('provincial.gpkg', 'catastroUR',    'catastro.pmtiles',      12, 16),
]
import fiona
for gpkg, *_ in PLAN:
    p = WORK / gpkg
    print(('OK ' if p.exists() else 'FALTA ') + gpkg,
          fiona.listlayers(p) if p.exists() else '')

In [ ]:
# 3) Tileado (réplica de tools/yvy_plantilla.py::gpkg_capa_a_pmtiles)
import geopandas as gpd, subprocess

def gpkg_a_pmtiles(gpkg, capa, salida, zmin, zmax, col_id='yvy_id'):
    gdf = gpd.read_file(gpkg, layer=capa)
    assert col_id in gdf.columns, f'{capa}: sin columna {col_id}'
    assert not gdf[col_id].isna().any(), f'{capa}: ids nulos'
    assert not gdf[col_id].duplicated().any(), f'{capa}: ids duplicados'
    if str(gdf.crs) != 'EPSG:4326':
        gdf = gdf.to_crs('EPSG:4326')
    gdf = gdf.rename(columns={col_id: 'id'})
    tmp = salida.parent / (salida.name + '.tmp.geojson')
    gdf.to_file(tmp, driver='GeoJSON')
    tmp_pm = salida.parent / (salida.name + '.tmp')
    cmd = ['tippecanoe', '-o', str(tmp_pm), '-Z', str(zmin), '-z', str(zmax),
           '-l', salida.stem, '--use-attribute-for-id=id',
           '--maximum-tile-bytes=500000', '--force', str(tmp)]
    r = subprocess.run(cmd, capture_output=True, text=True)
    tmp.unlink(missing_ok=True)
    assert r.returncode == 0, f'tippecanoe fallo: {r.stderr[-800:]}'
    assert tmp_pm.read_bytes()[:7] == b'PMTiles', 'magic invalido'
    tmp_pm.replace(salida)  # atomico
    print(f'OK {salida.name} ({salida.stat().st_size/1e6:.1f} MB, {len(gdf)} feats)')

for gpkg, capa, sal, zmin, zmax in PLAN:
    p = WORK / gpkg
    if not p.exists():
        print(f'SKIP {sal} (falta {gpkg})')
        continue
    gpkg_a_pmtiles(p, capa, OUT / sal, zmin, zmax)

In [ ]:
# 4) Verificacion: magic + header (min/max zoom) por archivo
import struct
for sal in sorted(OUT.glob('*.pmtiles')):
    h = sal.read_bytes()[:127]
    ok = h[:7] == b'PMTiles' and h[7] == 3
    zmin, zmax = h[100], h[101]
    print(f"{'OK' if ok else 'MAL'} {sal.name:24s} {sal.stat().st_size/1e6:7.1f} MB  z{zmin}-{zmax}")
print('\nSiguiente: copiar 08_pmtiles/*.pmtiles -> repo public/data/ + pnpm run build')